# Contextual Embedding Experiment

This notebook implements the contextual-input condition for the thesis project. It reuses the all-model isolated embedding pipeline, but each target word is represented by averaging its embeddings across generated Turkish sentence contexts.

The main analysis uses generated sentences where the target word appears exactly once as an unchanged lexical item under case-insensitive target matching. This admits ordinary sentence-initial capitalization such as `Adalet` for the target word `adalet`, while still excluding suffixed or otherwise changed forms. The original sentence casing is preserved for model input.


## Methodological note

For each contextual sentence, the model encodes the full sentence with its original casing. The target word span is located with a Turkish-aware, case-insensitive exact-match rule. Tokenizer offset mappings identify the model tokens that overlap this character span. The selected target subtokens are pooled inside the sentence with the same strategies used in the isolated condition: first, last, mean, and max.

For each target word, sentence-level vectors are averaged across all usable contexts. Pairwise cosine similarities and Spearman correlations are then computed against AnlamVer human similarity ratings. This makes the contextual results directly comparable with the earlier isolated-word outputs.


## Imports and paths

In [1]:
from pathlib import Path
import gc
import os
import random
import re
import tempfile

TEMP_CACHE_DIR = Path(tempfile.gettempdir()) / "thesis_project_cache"
MPLCONFIGDIR = TEMP_CACHE_DIR / "matplotlib"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))
os.environ.setdefault("XDG_CACHE_HOME", str(TEMP_CACHE_DIR))

import numpy as np
import pandas as pd
import torch
import matplotlib
try:
    get_ipython()
except NameError:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cosine as scipy_cosine_distance
from scipy.stats import spearmanr
from transformers import AutoModel, AutoTokenizer

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "anlamver-final.csv"
CONTEXT_SENTENCES_PATH = PROJECT_ROOT / "data" / "processed" / "context_sentences_llm.csv"
TOKENIZATION_PATH = PROJECT_ROOT / "data" / "processed" / "anlamver_tokenization_analysis.csv"
ISOLATED_SUMMARY_PATH = PROJECT_ROOT / "outputs" / "results" / "0302-isolated_all_models_summary.csv"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PAIR_SIMILARITIES_PATH = RESULTS_DIR / "0501-contextual_all_models_pair_similarities.csv"
SUMMARY_PATH = RESULTS_DIR / "0502-contextual_all_models_summary.csv"
ENRICHED_PAIR_SIMILARITIES_PATH = RESULTS_DIR / "0503-contextual_all_models_pair_similarities_with_tokenization.csv"
ISOLATED_VS_CONTEXTUAL_SUMMARY_PATH = RESULTS_DIR / "0504-isolated_vs_contextual_summary.csv"

MODEL_REGISTRY = {
    "BERTurk": "dbmdz/bert-base-turkish-cased",
    "mBERT": "bert-base-multilingual-cased",
    "XLM-R": "xlm-roberta-base",
}
LAYERS = [1, 7, 12]
POOLING_STRATEGIES = ["first", "last", "mean", "max"]
TURKISH_WORD_CHARS = r"A-Za-zÇĞİÖŞÜçğıöşü0-9_"
EXACT_MATCH_FLAGS = re.IGNORECASE

if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Context sentences: {CONTEXT_SENTENCES_PATH}")


/Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect
Device: mps
Context sentences: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/data/processed/context_sentences_llm.csv


## Load AnlamVer pairs and context sentences

In [2]:
def load_anlamver_pairs(path: Path) -> pd.DataFrame:
    pairs = pd.read_csv(path, encoding="cp1254", sep=";", decimal=",")
    required_columns = {"W1", "W2", "Sim"}
    missing_columns = required_columns - set(pairs.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    pairs = pairs.copy()
    pairs["W1"] = pairs["W1"].astype("string").str.strip()
    pairs["W2"] = pairs["W2"].astype("string").str.strip()
    pairs["Sim"] = pd.to_numeric(pairs["Sim"], errors="coerce")
    pairs = pairs.dropna(subset=["W1", "W2", "Sim"]).reset_index(drop=True)
    pairs = pairs[(pairs["W1"] != "") & (pairs["W2"] != "")].reset_index(drop=True)
    return pairs


def target_word_pattern(target_word: str, flags: int = EXACT_MATCH_FLAGS) -> re.Pattern:
    escaped_word = re.escape(target_word)
    return re.compile(rf"(?<![{TURKISH_WORD_CHARS}'’]){escaped_word}(?![{TURKISH_WORD_CHARS}'’])", flags)


def exact_match_count(sentence: str, target_word: str, flags: int = EXACT_MATCH_FLAGS) -> int:
    return len(list(target_word_pattern(target_word, flags=flags).finditer(sentence)))


def load_context_sentences(path: Path) -> pd.DataFrame:
    context_df = pd.read_csv(path)
    required_columns = {
        "target_word",
        "sentence_id",
        "sentence",
        "valid_exact_match",
        "exact_match_count",
    }
    missing_columns = required_columns - set(context_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required context-sentence columns: {sorted(missing_columns)}")

    context_df = context_df.copy()
    context_df["target_word"] = context_df["target_word"].astype("string").str.strip()
    context_df["sentence"] = context_df["sentence"].astype("string").str.strip()
    context_df = context_df.dropna(subset=["target_word", "sentence"]).reset_index(drop=True)
    context_df = context_df[(context_df["target_word"] != "") & (context_df["sentence"] != "")].reset_index(drop=True)
    context_df["valid_exact_match"] = context_df["valid_exact_match"].astype(bool)
    context_df["exact_match_count"] = pd.to_numeric(context_df["exact_match_count"], errors="coerce")
    context_df["case_sensitive_exact_match_count"] = context_df.apply(
        lambda row: exact_match_count(row["sentence"], row["target_word"], flags=0), axis=1
    )
    context_df["case_insensitive_exact_match_count"] = context_df.apply(
        lambda row: exact_match_count(row["sentence"], row["target_word"]), axis=1
    )
    context_df["case_insensitive_exact_match"] = context_df["case_insensitive_exact_match_count"].eq(1)
    return context_df


pairs = load_anlamver_pairs(DATA_PATH)
all_context_sentences = load_context_sentences(CONTEXT_SENTENCES_PATH)
context_sentences = all_context_sentences.loc[
    all_context_sentences["case_insensitive_exact_match"]
].copy()
case_sensitive_context_sentences = all_context_sentences.loc[
    all_context_sentences["case_sensitive_exact_match_count"].eq(1)
]
capitalization_recovered_rows = len(context_sentences) - len(case_sensitive_context_sentences)

pair_words = sorted(pd.concat([pairs["W1"], pairs["W2"]]).unique())
context_sentence_counts = context_sentences.groupby("target_word").size()
words_without_context = sorted(set(pair_words) - set(context_sentence_counts.index))

# Every selected sentence should satisfy the case-insensitive exact-match policy before inference starts.
assert context_sentences["case_insensitive_exact_match"].all()
assert context_sentences["case_insensitive_exact_match_count"].eq(1).all()

print(f"Loaded {len(pairs):,} AnlamVer pairs")
print(f"Loaded {len(all_context_sentences):,} generated context rows")
print(f"Using {len(context_sentences):,} case-insensitive exact-match context rows")
print(f"Rows recovered by case-insensitive matching: {capitalization_recovered_rows:,}")
print(f"Words with at least one case-insensitive exact-match context: {context_sentences['target_word'].nunique():,}/{len(pair_words):,}")
print(f"Pair words without context: {len(words_without_context):,}")

context_sentence_counts.describe()


Loaded 500 AnlamVer pairs
Loaded 4,755 generated context rows
Using 4,684 case-insensitive exact-match context rows
Rows recovered by case-insensitive matching: 1,411
Words with at least one case-insensitive exact-match context: 317/317
Pair words without context: 0


count    317.000000
mean      14.776025
std        0.813451
min        8.000000
25%       15.000000
50%       15.000000
75%       15.000000
max       15.000000
dtype: float64

## Target span and pooling helpers

In [3]:
WORD_TOKEN_PATTERN = re.compile(rf"[{TURKISH_WORD_CHARS}]+(?:['’][{TURKISH_WORD_CHARS}]+)?")


def find_target_span(sentence: str, target_word: str) -> tuple[int, int]:
    matches = list(target_word_pattern(target_word).finditer(sentence))
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one target match for {target_word!r}, found {len(matches)} in: {sentence!r}"
        )
    match = matches[0]
    return match.start(), match.end()


def pool_first(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    return subtoken_embeddings[0]


def pool_last(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    return subtoken_embeddings[-1]


def pool_mean(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    return subtoken_embeddings.mean(dim=0)


def pool_max(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    return subtoken_embeddings.max(dim=0).values


POOLING_FUNCTIONS = {
    "first": pool_first,
    "last": pool_last,
    "mean": pool_mean,
    "max": pool_max,
}


def cosine_similarity_np(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    vec_a = np.asarray(vec_a, dtype=float)
    vec_b = np.asarray(vec_b, dtype=float)
    if not np.isfinite(vec_a).all() or not np.isfinite(vec_b).all():
        return np.nan
    if np.linalg.norm(vec_a) == 0 or np.linalg.norm(vec_b) == 0:
        return np.nan
    return float(1 - scipy_cosine_distance(vec_a, vec_b))


## Model and contextual extraction functions

In [4]:
def load_model_and_tokenizer(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if not getattr(tokenizer, "is_fast", False):
        raise ValueError(f"Fast tokenizer with offset mapping is required for {model_name}")
    model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
    model.config.output_hidden_states = True
    model.to(DEVICE)
    model.eval()
    return tokenizer, model


def token_indices_for_span(offset_mapping, special_tokens_mask, target_span: tuple[int, int]) -> list[int]:
    target_start, target_end = target_span
    selected_indices = []
    for index, ((token_start, token_end), is_special) in enumerate(zip(offset_mapping, special_tokens_mask)):
        if is_special or token_start == token_end:
            continue
        overlaps_target = token_start < target_end and token_end > target_start
        if overlaps_target:
            selected_indices.append(index)
    return selected_indices


@torch.no_grad()
def extract_contextual_sentence_embeddings(
    sentence: str,
    target_word: str,
    tokenizer,
    model,
    layers: list[int],
    pooling_functions: dict[str, callable],
    device: torch.device,
) -> dict[int, dict[str, np.ndarray]]:
    target_span = find_target_span(sentence, target_word)
    encoded = tokenizer(
        sentence,
        add_special_tokens=True,
        return_tensors="pt",
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
    )
    offset_mapping = encoded.pop("offset_mapping")[0].detach().cpu().tolist()
    special_tokens_mask = encoded.pop("special_tokens_mask")[0].detach().cpu().tolist()
    target_token_indices = token_indices_for_span(offset_mapping, special_tokens_mask, target_span)
    if not target_token_indices:
        tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0].detach().cpu().tolist())
        raise ValueError(
            f"No tokenizer tokens overlap target span {target_span} for {target_word!r} in {sentence!r}. "
            f"Tokens: {tokens}; offsets: {offset_mapping}"
        )

    encoded = {key: value.to(device) for key, value in encoded.items()}
    outputs = model(**encoded)
    hidden_states = outputs.hidden_states
    max_layer = len(hidden_states) - 1
    invalid_layers = [layer for layer in layers if layer < 1 or layer > max_layer]
    if invalid_layers:
        raise ValueError(f"Invalid layers {invalid_layers}; model provides layers 1..{max_layer}")

    sentence_embeddings = {}
    for layer in layers:
        layer_embeddings = hidden_states[layer][0, target_token_indices, :]
        sentence_embeddings[layer] = {
            pooling_name: pooling_fn(layer_embeddings).detach().cpu().float().numpy()
            for pooling_name, pooling_fn in pooling_functions.items()
        }
    return sentence_embeddings


def average_contextual_embeddings(
    sentence_embeddings: list[dict[int, dict[str, np.ndarray]]],
    layers: list[int],
    pooling_strategies: list[str],
) -> dict[int, dict[str, np.ndarray]]:
    averaged_embeddings = {}
    for layer in layers:
        averaged_embeddings[layer] = {}
        for pooling in pooling_strategies:
            vectors = np.stack([embedding[layer][pooling] for embedding in sentence_embeddings], axis=0)
            averaged_embeddings[layer][pooling] = vectors.mean(axis=0)
    return averaged_embeddings


def build_contextual_embedding_cache(
    context_df: pd.DataFrame,
    tokenizer,
    model,
    layers: list[int],
    pooling_functions: dict[str, callable],
    device: torch.device,
) -> tuple[dict[str, dict[int, dict[str, np.ndarray]]], dict[str, int]]:
    embeddings = {}
    sentence_counts = {}
    grouped = list(context_df.groupby("target_word", sort=True))
    for index, (target_word, group) in enumerate(grouped, start=1):
        sentence_embedding_list = []
        for sentence in group["sentence"].astype(str):
            sentence_embedding_list.append(
                extract_contextual_sentence_embeddings(
                    sentence=sentence,
                    target_word=str(target_word),
                    tokenizer=tokenizer,
                    model=model,
                    layers=layers,
                    pooling_functions=pooling_functions,
                    device=device,
                )
            )
        embeddings[str(target_word)] = average_contextual_embeddings(
            sentence_embeddings=sentence_embedding_list,
            layers=layers,
            pooling_strategies=list(pooling_functions),
        )
        sentence_counts[str(target_word)] = len(sentence_embedding_list)
        if index % 25 == 0 or index == len(grouped):
            print(f"Averaged contexts for {index:>3}/{len(grouped)} words")
    return embeddings, sentence_counts


## Pair similarity and evaluation functions

In [5]:
def compute_pair_similarities(
    pairs: pd.DataFrame,
    embeddings: dict[str, dict[int, dict[str, np.ndarray]]],
    context_counts: dict[str, int],
    model_key: str,
    layers: list[int],
    pooling_strategies: list[str],
) -> pd.DataFrame:
    id_columns = [column for column in ["QID", "W1", "W2", "Sim"] if column in pairs.columns]
    rows = []
    for _, row in pairs.iterrows():
        word_1 = str(row["W1"])
        word_2 = str(row["W2"])
        word_1_embeddings = embeddings.get(word_1)
        word_2_embeddings = embeddings.get(word_2)
        for layer in layers:
            for pooling in pooling_strategies:
                if word_1_embeddings is None or word_2_embeddings is None:
                    cosine_similarity = np.nan
                else:
                    cosine_similarity = cosine_similarity_np(
                        word_1_embeddings[layer][pooling],
                        word_2_embeddings[layer][pooling],
                    )
                rows.append(
                    {
                        **{column: row[column] for column in id_columns},
                        "input_condition": "contextual",
                        "model": model_key,
                        "layer": layer,
                        "pooling": pooling,
                        "cosine_similarity": cosine_similarity,
                        "word1_context_sentence_count": context_counts.get(word_1, 0),
                        "word2_context_sentence_count": context_counts.get(word_2, 0),
                    }
                )
    return pd.DataFrame(rows)


def summarize_spearman(pair_similarity_df: pd.DataFrame, human_column: str = "Sim") -> pd.DataFrame:
    rows = []
    group_columns = ["input_condition", "model", "layer", "pooling"]
    for (input_condition, model_key, layer, pooling), group in pair_similarity_df.groupby(group_columns, sort=False):
        valid = group[[human_column, "cosine_similarity"]].dropna()
        if len(valid) < 3 or valid[human_column].nunique() < 2 or valid["cosine_similarity"].nunique() < 2:
            rho, p_value = np.nan, np.nan
        else:
            rho, p_value = spearmanr(valid[human_column], valid["cosine_similarity"])
        rows.append(
            {
                "input_condition": input_condition,
                "model": model_key,
                "layer": layer,
                "pooling": pooling,
                "spearman_rho": rho,
                "p_value": p_value,
                "n_pairs": len(valid),
            }
        )
    return pd.DataFrame(rows)


def sort_experiment_results(df: pd.DataFrame) -> pd.DataFrame:
    model_order = {model_key: index for index, model_key in enumerate(MODEL_REGISTRY)}
    pooling_order = {pooling: index for index, pooling in enumerate(POOLING_STRATEGIES)}
    sorted_df = df.assign(
        _model_order=df["model"].map(model_order),
        _pooling_order=df["pooling"].map(pooling_order),
    )
    sort_columns = ["_model_order", "layer", "_pooling_order"]
    if "QID" in sorted_df.columns:
        sort_columns.append("QID")
    sorted_df = sorted_df.sort_values(sort_columns)
    return sorted_df.drop(columns=["_model_order", "_pooling_order"]).reset_index(drop=True)


## Sanity check

In [6]:
SANITY_MODEL_KEY = "XLM-R"
SANITY_WORD = "adalet" if "adalet" in set(context_sentences["target_word"]) else str(context_sentences.iloc[0]["target_word"])
SANITY_SENTENCE = str(context_sentences.loc[context_sentences["target_word"].eq(SANITY_WORD), "sentence"].iloc[0])

sanity_tokenizer, sanity_model = load_model_and_tokenizer(MODEL_REGISTRY[SANITY_MODEL_KEY])
sanity_embeddings = extract_contextual_sentence_embeddings(
    sentence=SANITY_SENTENCE,
    target_word=SANITY_WORD,
    tokenizer=sanity_tokenizer,
    model=sanity_model,
    layers=LAYERS,
    pooling_functions=POOLING_FUNCTIONS,
    device=DEVICE,
)

sanity_rows = []
for layer, pooling_dict in sanity_embeddings.items():
    for pooling_name, vector in pooling_dict.items():
        sanity_rows.append(
            {
                "model": SANITY_MODEL_KEY,
                "target_word": SANITY_WORD,
                "sentence": SANITY_SENTENCE,
                "layer": layer,
                "pooling": pooling_name,
                "shape": vector.shape,
                "finite_values": bool(np.isfinite(vector).all()),
                "l2_norm": float(np.linalg.norm(vector)),
            }
        )

sanity_df = pd.DataFrame(sanity_rows)
assert set(sanity_embeddings.keys()) == set(LAYERS)
assert all(set(sanity_embeddings[layer].keys()) == set(POOLING_STRATEGIES) for layer in LAYERS)
assert sanity_df["finite_values"].all()
same_vector_cosine = cosine_similarity_np(sanity_embeddings[7]["mean"], sanity_embeddings[7]["mean"])
assert np.isclose(same_vector_cosine, 1.0, atol=1e-6)

print(f"Sanity sentence: {SANITY_SENTENCE}")
print(f"Target word: {SANITY_WORD}")
print(f"Same-vector cosine: {same_vector_cosine:.6f}")

del sanity_embeddings, sanity_model, sanity_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
if hasattr(torch, "mps") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    torch.mps.empty_cache()
gc.collect()

sanity_df


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10032.77it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sanity sentence: Mahkeme kararıyla adalet bir kez daha tecelli etti.
Target word: adalet
Same-vector cosine: 1.000000


,model,target_word,sentence,layer,pooling,shape,finite_values,l2_norm
0,XLM-R,adalet,Mahkeme kararıyla adalet bir kez daha tecelli etti.,1,first,"(768,)",True,13.851565
1,XLM-R,adalet,Mahkeme kararıyla adalet bir kez daha tecelli etti.,1,last,"(768,)",True,14.028101
2,XLM-R,adalet,Mahkeme kararıyla adalet bir kez daha tecelli etti.,1,mean,"(768,)",True,11.181485
3,XLM-R,adalet,Mahkeme kararıyla adalet bir kez daha tecelli etti.,1,max,"(768,)",True,13.424209
4,XLM-R,adalet,Mahkeme kararıyla adalet bir kez daha tecelli etti.,7,first,"(768,)",True,22.237848
5,XLM-R,adalet,Mahkeme kararıyla adalet bir kez daha tecelli etti.,7,last,"(768,)",True,22.822327
6,XLM-R,adalet,Mahkeme kararıyla adalet bir kez daha tecelli etti.,7,mean,"(768,)",True,21.612305
7,XLM-R,adalet,Mahkeme kararıyla adalet bir kez daha tecelli etti.,7,max,"(768,)",True,22.192213
8,XLM-R,adalet,Mahkeme kararıyla adalet bir kez daha tecelli etti.,12,first,"(768,)",True,19.390053
9,XLM-R,adalet,Mahkeme kararıyla adalet bir kez daha tecelli etti.,12,last,"(768,)",True,19.423498


## Run full contextual all-model experiment

In [7]:
def run_model_contextual_experiment(
    model_key: str,
    model_name: str,
    pairs: pd.DataFrame,
    context_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    print("=" * 80)
    print(f"Running {model_key}: {model_name}")
    tokenizer, model = load_model_and_tokenizer(model_name)
    print(f"Tokenizer: {type(tokenizer).__name__}")
    print(f"Model: {type(model).__name__}")

    embeddings, sentence_counts = build_contextual_embedding_cache(
        context_df=context_df,
        tokenizer=tokenizer,
        model=model,
        layers=LAYERS,
        pooling_functions=POOLING_FUNCTIONS,
        device=DEVICE,
    )
    pair_similarity_df = compute_pair_similarities(
        pairs=pairs,
        embeddings=embeddings,
        context_counts=sentence_counts,
        model_key=model_key,
        layers=LAYERS,
        pooling_strategies=POOLING_STRATEGIES,
    )
    pair_similarity_df["model_name"] = model_name
    summary_df = summarize_spearman(pair_similarity_df, human_column="Sim")
    summary_df["model_name"] = model_name

    del embeddings, sentence_counts, model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch, "mps") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()

    return pair_similarity_df, summary_df


all_pair_similarity_frames = []
all_summary_frames = []

for model_key, model_name in MODEL_REGISTRY.items():
    pair_similarity_df, summary_df = run_model_contextual_experiment(
        model_key=model_key,
        model_name=model_name,
        pairs=pairs,
        context_df=context_sentences,
    )
    all_pair_similarity_frames.append(pair_similarity_df)
    all_summary_frames.append(summary_df)

all_pair_similarities = sort_experiment_results(pd.concat(all_pair_similarity_frames, ignore_index=True))
all_summary = sort_experiment_results(pd.concat(all_summary_frames, ignore_index=True))

expected_rows = len(pairs) * len(MODEL_REGISTRY) * len(LAYERS) * len(POOLING_STRATEGIES)
assert len(all_pair_similarities) == expected_rows
summary_counts = (
    all_pair_similarities.groupby(["input_condition", "model", "layer", "pooling"])["cosine_similarity"]
    .apply(lambda s: int(s.notna().sum()))
    .reset_index(name="expected_n_pairs")
)
summary_check = all_summary.merge(summary_counts, on=["input_condition", "model", "layer", "pooling"], how="left")
assert summary_check["n_pairs"].eq(summary_check["expected_n_pairs"]).all()

print(f"Pair-level rows: {len(all_pair_similarities):,}")
print(f"Summary rows: {len(all_summary):,}")
all_summary.sort_values("spearman_rho", ascending=False).head(12)


Running BERTurk: dbmdz/bert-base-turkish-cased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7457.31it/s]
[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer: BertTokenizer
Model: BertModel
Averaged contexts for  25/317 words
Averaged contexts for  50/317 words
Averaged contexts for  75/317 words
Averaged contexts for 100/317 words
Averaged contexts for 125/317 words
Averaged contexts for 150/317 words
Averaged contexts for 175/317 words
Averaged contexts for 200/317 words
Averaged contexts for 225/317 words
Averaged contexts for 250/317 words
Averaged contexts for 275/317 words
Averaged contexts for 300/317 words
Averaged contexts for 317/317 words
Running mBERT: bert-base-multilingual-cased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8390.13it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer: BertTokenizer
Model: BertModel
Averaged contexts for  25/317 words
Averaged contexts for  50/317 words
Averaged contexts for  75/317 words
Averaged contexts for 100/317 words
Averaged contexts for 125/317 words
Averaged contexts for 150/317 words
Averaged contexts for 175/317 words
Averaged contexts for 200/317 words
Averaged contexts for 225/317 words
Averaged contexts for 250/317 words
Averaged contexts for 275/317 words
Averaged contexts for 300/317 words
Averaged contexts for 317/317 words
Running XLM-R: xlm-roberta-base


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7767.37it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer: XLMRobertaTokenizer
Model: XLMRobertaModel
Averaged contexts for  25/317 words
Averaged contexts for  50/317 words
Averaged contexts for  75/317 words
Averaged contexts for 100/317 words
Averaged contexts for 125/317 words
Averaged contexts for 150/317 words
Averaged contexts for 175/317 words
Averaged contexts for 200/317 words
Averaged contexts for 225/317 words
Averaged contexts for 250/317 words
Averaged contexts for 275/317 words
Averaged contexts for 300/317 words
Averaged contexts for 317/317 words
Pair-level rows: 18,000
Summary rows: 36


,input_condition,model,layer,pooling,spearman_rho,p_value,n_pairs,model_name
4,contextual,BERTurk,7,first,0.527538,3.630230e-37,500,dbmdz/bert-base-turkish-cased
0,contextual,BERTurk,1,first,0.455468,5.608269e-27,500,dbmdz/bert-base-turkish-cased
8,contextual,BERTurk,12,first,0.452649,1.259832e-26,500,dbmdz/bert-base-turkish-cased
10,contextual,BERTurk,12,mean,0.450441,2.361642e-26,500,dbmdz/bert-base-turkish-cased
6,contextual,BERTurk,7,mean,0.426034,1.822955e-23,500,dbmdz/bert-base-turkish-cased
5,contextual,BERTurk,7,last,0.413901,4.085490e-22,500,dbmdz/bert-base-turkish-cased
9,contextual,BERTurk,12,last,0.407720,1.897754e-21,500,dbmdz/bert-base-turkish-cased
11,contextual,BERTurk,12,max,0.352732,4.279765e-16,500,dbmdz/bert-base-turkish-cased
7,contextual,BERTurk,7,max,0.337657,8.465374e-15,500,dbmdz/bert-base-turkish-cased
1,contextual,BERTurk,1,last,0.329436,4.029831e-14,500,dbmdz/bert-base-turkish-cased


## Save contextual outputs

In [ ]:
all_pair_similarities.to_csv(PAIR_SIMILARITIES_PATH, index=False)
all_summary.to_csv(SUMMARY_PATH, index=False)

TOKENIZATION_PREFIX_BY_MODEL = {
    "BERTurk": "berturk",
    "mBERT": "mbert",
    "XLM-R": "xlmr",
}


def build_model_specific_complexity_table(tokenization_df: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for model_key, prefix in TOKENIZATION_PREFIX_BY_MODEL.items():
        rename_map = {
            f"{prefix}_pair_total_subtoken_count": "pair_total_subtoken_count",
            f"{prefix}_pair_max_subtoken_count": "pair_max_subtoken_count",
            f"{prefix}_pair_clean": "clean_pair",
            f"{prefix}_pair_split": "split_pair",
            f"{prefix}_pair_word1_is_split": "word1_is_split",
            f"{prefix}_pair_word2_is_split": "word2_is_split",
            f"{prefix}_pair_both_words_split": "both_words_split",
            f"{prefix}_pair_avg_subtoken_char_length": "pair_avg_subtoken_char_length",
            f"{prefix}_word1_avg_subtoken_char_length": "word1_avg_subtoken_char_length",
            f"{prefix}_word2_avg_subtoken_char_length": "word2_avg_subtoken_char_length",
            f"{prefix}_word1_subtoken_count_per_char": "word1_subtoken_count_per_char",
            f"{prefix}_word2_subtoken_count_per_char": "word2_subtoken_count_per_char",
        }
        available_columns = ["QID", *[column for column in rename_map if column in tokenization_df.columns]]
        model_complexity = tokenization_df[available_columns].rename(columns=rename_map).copy()
        model_complexity["model"] = model_key
        frames.append(model_complexity)
    return pd.concat(frames, ignore_index=True)


tokenization_features = pd.read_csv(TOKENIZATION_PATH)
model_tokenization_complexity = build_model_specific_complexity_table(tokenization_features)
enriched_pair_similarities = all_pair_similarities.merge(
    model_tokenization_complexity,
    on=["QID", "model"],
    how="left",
    validate="many_to_one",
)
enriched_pair_similarities.to_csv(ENRICHED_PAIR_SIMILARITIES_PATH, index=False)

isolated_summary = pd.read_csv(ISOLATED_SUMMARY_PATH)
isolated_summary = isolated_summary.copy()
isolated_summary.insert(0, "input_condition", "isolated")
combined_summary = pd.concat([isolated_summary, all_summary], ignore_index=True, sort=False)
combined_summary = sort_experiment_results(combined_summary)
combined_summary.to_csv(ISOLATED_VS_CONTEXTUAL_SUMMARY_PATH, index=False)

print(f"Saved pair-level contextual similarities to: {PAIR_SIMILARITIES_PATH}")
print(f"Saved contextual summary to: {SUMMARY_PATH}")
print(f"Saved enriched contextual similarities to: {ENRICHED_PAIR_SIMILARITIES_PATH}")
print(f"Saved isolated/contextual comparison to: {ISOLATED_VS_CONTEXTUAL_SUMMARY_PATH}")
combined_summary.sort_values("spearman_rho", ascending=False).head(12)


## Isolated vs contextual figures

In [ ]:
def save_current_figure(filename: str) -> None:
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved figure to: {path}")


best_by_model_condition = (
    combined_summary.sort_values("spearman_rho", ascending=False)
    .groupby(["input_condition", "model"], as_index=False)
    .head(1)
)

plt.figure(figsize=(7, 4))
sns.barplot(
    data=best_by_model_condition,
    x="model",
    y="spearman_rho",
    hue="input_condition",
    hue_order=["isolated", "contextual"],
)
plt.title("Best Spearman by model and input condition")
plt.xlabel("Model")
plt.ylabel("Spearman rho")
plt.ylim(0, max(0.05, best_by_model_condition["spearman_rho"].max() + 0.05))
save_current_figure("0501-isolated_vs_contextual_best_by_model.png")

contextual_heatmap = all_summary.pivot_table(
    index="model", columns="pooling", values="spearman_rho", aggfunc="max"
)[POOLING_STRATEGIES]
plt.figure(figsize=(8, 3.6))
sns.heatmap(contextual_heatmap, annot=True, fmt=".3f", cmap="Greens", linewidths=0.5)
plt.title("Contextual best Spearman by model and pooling strategy")
plt.xlabel("Pooling strategy")
plt.ylabel("Model")
save_current_figure("0502-contextual_model_pooling_spearman_heatmap.png")

contextual_full_heatmap = (
    all_summary.assign(model_layer=lambda df_: df_["model"] + " L" + df_["layer"].astype(str))
    .pivot_table(index="model_layer", columns="pooling", values="spearman_rho", aggfunc="first")
    [POOLING_STRATEGIES]
)
plt.figure(figsize=(8, 6))
sns.heatmap(contextual_full_heatmap, annot=True, fmt=".3f", cmap="YlGnBu", linewidths=0.5)
plt.title("Contextual Spearman for every model-layer-pooling configuration")
plt.xlabel("Pooling strategy")
plt.ylabel("Model and layer")
save_current_figure("0503-contextual_full_configuration_spearman_heatmap.png")

comparison_grid = (
    combined_summary.assign(config=lambda df_: df_["model"] + " L" + df_["layer"].astype(str) + " " + df_["pooling"])
    .pivot_table(index="config", columns="input_condition", values="spearman_rho", aggfunc="first")
    .dropna()
)
comparison_grid["contextual_minus_isolated"] = comparison_grid["contextual"] - comparison_grid["isolated"]
comparison_grid = comparison_grid.sort_values("contextual_minus_isolated", ascending=False)

plt.figure(figsize=(8, 8))
sns.barplot(
    data=comparison_grid.reset_index(),
    y="config",
    x="contextual_minus_isolated",
    color="#4C78A8",
)
plt.axvline(0, color="black", linewidth=1)
plt.title("Contextual minus isolated Spearman by configuration")
plt.xlabel("Spearman rho difference")
plt.ylabel("Configuration")
save_current_figure("0504-contextual_minus_isolated_by_configuration.png")

comparison_grid.head(12)


## Best results and interpretation notes

In [ ]:
print("Best contextual configurations")
display(all_summary.sort_values("spearman_rho", ascending=False).head(12))

print("Best isolated/contextual configurations")
display(combined_summary.sort_values("spearman_rho", ascending=False).head(12))

print("Context sentence coverage by pair")
coverage_columns = ["QID", "W1", "W2", "word1_context_sentence_count", "word2_context_sentence_count"]
display(
    all_pair_similarities[coverage_columns]
    .drop_duplicates()
    .assign(pair_has_both_contexts=lambda df_: (df_["word1_context_sentence_count"] > 0) & (df_["word2_context_sentence_count"] > 0))
    ["pair_has_both_contexts"]
    .value_counts()
)
